# Macaulay Equity Duration (Price-Implied Terminal Value)


## 1) Load data


In [28]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)

# Prefer local dataset filename; fallback to existing Project_Data layout
input_candidates = [
    Path('euro500_macaulay.parquet'),
    Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_macaulay.parquet'),
]
INPUT_PATH = next((p for p in input_candidates if p.exists()), None)
if INPUT_PATH is None:
    raise FileNotFoundError('Could not find euro500_macaulay.parquet in expected locations.')

DATA_DIR = Path('/Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate')
OUTPUT_PATH = DATA_DIR / 'EQDuration_Macaulay.parquet'

df = pd.read_parquet(INPUT_PATH)
print(f'Loaded rows: {len(df):,}')
print(f'Loaded columns: {len(df.columns):,}')
print(f'Input file: {INPUT_PATH.resolve()}')


Loaded rows: 56,500
Loaded columns: 33
Input file: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/euro500_macaulay.parquet


In [29]:
# Required fields for this model
required_cols = [
    'PriceClose',
    'EPS_FY1',
    'EPS_FY2',
    'EPS_FY3',
]

# Required data cleaning
# - drop missing PriceClose / EPS_FY1 / EPS_FY2 / EPS_FY3
# - drop PriceClose <= 0 and EPS_FY1 <= 0
# - do NOT drop based on EPS_FY4 / EPS_FY5 missingness
df = df.dropna(subset=required_cols).copy()
df = df[(df['PriceClose'] > 0) & (df['EPS_FY1'] > 0)].copy()

print(f'Rows after required cleaning: {len(df):,}')


Rows after required cleaning: 21,893


## 2) Discount rate


In [30]:
# Stable constant discount rate
R = 0.10
G_INF = 0.02

df['r'] = R
print(f'Using constant discount rate r = {R:.2%}')


Using constant discount rate r = 10.00%


## 3) LTG preparation


In [31]:
# LTG is used only as medium-run transition signal
# Convert percent to decimal and clip to economically plausible range
df['LTG_decimal'] = pd.to_numeric(df['LTG'], errors='coerce') / 100.0
df['g0'] = df['LTG_decimal'].clip(lower=-0.02, upper=0.15)

# Neutral fallback when LTG is missing: start transition at long-run growth
df['g0'] = df['g0'].fillna(G_INF)


## 4) Explicit cashflows (years 1-3)


In [32]:
df['C1'] = pd.to_numeric(df['EPS_FY1'], errors='coerce')
df['C2'] = pd.to_numeric(df['EPS_FY2'], errors='coerce')
df['C3'] = pd.to_numeric(df['EPS_FY3'], errors='coerce')


## 5) Transition growth path (years 4-10)


In [33]:
# Growth fades linearly from g0 in year 4 to g_inf in year 10
for t in range(4, 11):
    df[f'g_{t}'] = df['g0'] + ((t - 4) / (10 - 4)) * (G_INF - df['g0'])

# Recursive transition cashflow construction
df['C4'] = df['C3'] * (1.0 + df['g_4'])
for t in range(5, 11):
    df[f'C{t}'] = df[f'C{t-1}'] * (1.0 + df[f'g_{t}'])


## 6) Price-Implied Terminal Block (No Gordon TV)


In [34]:
# Present value of explicit cashflows only (t=1..10)
for t in range(1, 11):
    df[f'PV_{t}'] = df[f'C{t}'] / np.power(1.0 + df['r'], t)

pv_cols = [f'PV_{t}' for t in range(1, 11)]
df['PV_explicit'] = df[pv_cols].sum(axis=1)

# Terminal block is residual present value implied by observed price
df['PV_terminal'] = df['PriceClose'] - df['PV_explicit']

# By construction anchor total valuation to observed stock price
df['PV_total'] = df['PriceClose']


## 7) Macaulay Duration with Price Anchor


In [35]:
# Terminal maturity of level perpetuity beginning in year 10
# T_terminal = 10 + (1 + r) / r = 21 when r=10%
df['T_terminal'] = 10.0 + (1.0 + df['r']) / df['r']

# Duration numerator: explicit block + price-implied terminal block
duration_num = np.zeros(len(df), dtype=float)
for t in range(1, 11):
    duration_num += t * df[f'PV_{t}'].to_numpy()

duration_num += (df['T_terminal'] * df['PV_terminal']).to_numpy()
df['Duration_num'] = duration_num

df['duration_macaulay'] = df['Duration_num'] / df['PriceClose']


## 8) Safeguards and Cleaning


In [36]:
# Requested diagnostics flags
df['flag_negative_terminal'] = df['PV_terminal'] < 0
df['flag_small_price'] = df['PriceClose'] <= 1
df['flag_bad_duration'] = (df['duration_macaulay'] <= 0) | (df['duration_macaulay'] > 100)

# Clean duration version
df['duration_macaulay_clean'] = df['duration_macaulay']
invalid_mask = df['flag_negative_terminal'] | df['flag_small_price'] | df['flag_bad_duration']
df.loc[invalid_mask, 'duration_macaulay_clean'] = np.nan

# Winsorize cleaned duration at 1% and 99%
q01 = df['duration_macaulay_clean'].quantile(0.01)
q99 = df['duration_macaulay_clean'].quantile(0.99)
df['duration_macaulay_clean'] = df['duration_macaulay_clean'].clip(lower=q01, upper=q99)

print(f'Winsor bounds on cleaned duration: [{q01:.6f}, {q99:.6f}]')


Winsor bounds on cleaned duration: [5.213471, 18.055910]


## 9) Diagnostics


In [37]:
# A) Raw duration summary
print('A) duration_macaulay describe()')
print(df['duration_macaulay'].describe())
print()

# B) Cleaned duration summary
print('B) duration_macaulay_clean describe()')
print(df['duration_macaulay_clean'].describe())
print()

# C) Share negative terminal residual
share_negative_terminal = df['flag_negative_terminal'].mean()
print(f'C) Share negative terminal residual: {share_negative_terminal:.4%}')

# D) Share duration > 100 before cleaning
share_duration_gt_100 = (df['duration_macaulay'] > 100).mean()
print(f'D) Share duration > 100 before cleaning: {share_duration_gt_100:.4%}')

# E) Share retained after cleaning
share_retained = df['duration_macaulay_clean'].notna().mean()
print(f'E) Share retained after cleaning: {share_retained:.4%}')
print()

# F) Cross-sectional std by quarter/date
if 'date' in df.columns:
    cs_std_date = df.groupby('date', dropna=False)['duration_macaulay_clean'].std()
    print('F) Cross-sectional std by date (describe):')
    print(cs_std_date.describe())
    print()

if 'quarter' in df.columns:
    cs_std_quarter = df.groupby('quarter', dropna=False)['duration_macaulay_clean'].std()
    print('F) Cross-sectional std by quarter (describe):')
    print(cs_std_quarter.describe())
    print()

# G) Quintile median durations
valid = df['duration_macaulay_clean'].dropna()
if len(valid) >= 5:
    q_labels = [1, 2, 3, 4, 5]
    df.loc[valid.index, 'duration_quintile'] = pd.qcut(valid, q=5, labels=q_labels)
    q_medians = df.groupby('duration_quintile', observed=False)['duration_macaulay_clean'].median()
    print('G) Quintile median durations (cleaned):')
    print(q_medians)
    print()

# Explicit-horizon valuation share diagnostics
print('(PV_explicit / PriceClose).describe()')
print((df['PV_explicit'] / df['PriceClose']).describe())


A) duration_macaulay describe()
count         21893.0
mean        -2.663344
std        225.030698
min     -22255.845343
25%         -1.980825
50%           5.82534
75%         10.766629
max         74.631639
Name: duration_macaulay, dtype: Float64

B) duration_macaulay_clean describe()
count      11552.0
mean     10.656333
std       3.212385
min       5.213471
25%       8.045518
50%      10.510642
75%      13.092822
max       18.05591
Name: duration_macaulay_clean, dtype: Float64

C) Share negative terminal residual: 47.1566%
D) Share duration > 100 before cleaning: 0.0000%
E) Share retained after cleaning: 52.7657%

F) Cross-sectional std by date (describe):
count        89.0
mean      3.02625
std      0.337686
min       2.28774
25%       2.81173
50%      3.005322
75%      3.248896
max      3.862082
Name: duration_macaulay_clean, dtype: Float64

F) Cross-sectional std by quarter (describe):
count        89.0
mean      3.02625
std      0.337686
min       2.28774
25%       2.81173
50%  

## 10) Interpretation Checks


### Why the old Gordon terminal value exploded
With a Gordon terminal block, duration can blow up when the denominator `(r - g_inf)` is small or when growth assumptions imply very large continuation values. That pushes too much PV mass to far-dated cashflows and mechanically inflates Macaulay duration.

### Why the price-implied residual stabilizes the measure
Using `PV_terminal = PriceClose - PV_explicit` forces total PV to match the observed stock price exactly. The terminal block is therefore bounded by market valuation, avoiding an unconstrained model-implied terminal explosion.

### Why this is closer to implied equity duration logic
The construction keeps an explicit forecast horizon (years 1-10) and infers the continuation block from the price residual, which aligns with implied-duration designs in the Dechow-style spirit: explicit near-term fundamentals plus market-implied long-run value.

### Why this is better for ECB-shock regressions
A price-anchored duration is cross-sectionally more stable and less dominated by tail Gordon assumptions. That improves interpretability and reduces mechanical outliers in rate-sensitivity regressions.


## 11) Output


In [38]:
# Save final dataframe including requested columns and all existing fields
required_new_cols = [
    'duration_macaulay',
    'duration_macaulay_clean',
    'PV_explicit',
    'PV_terminal',
    'flag_negative_terminal',
    'flag_bad_duration',
]

missing_new = [c for c in required_new_cols if c not in df.columns]
if missing_new:
    raise ValueError(f'Missing required output columns: {missing_new}')

DATA_DIR.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH.resolve()}')
print(f'Rows saved: {len(df):,}')
print('Saved fields include duration_macaulay, duration_macaulay_clean, PV_explicit, PV_terminal, and flags.')


Saved: /Users/jakob/Documents/Parforceleistung/Studium/MSc Economics/Masterarbeit/Project_Data/intermediate/EQDuration_Macaulay.parquet
Rows saved: 21,893
Saved fields include duration_macaulay, duration_macaulay_clean, PV_explicit, PV_terminal, and flags.
